In [ ]:
!pip install transformers
import torch

## The DistilGPT2 model
DistilGPT2 is a lite version of the OpenAI's GPT-2 model. It was developed by Huggingface using knowledge distillation, where a larger "teacher" model (GPT-2) trains this smaller "student" to mimic its behavior.
In the words of Huggingface,
>> DistilGPT2 (short for Distilled-GPT2) is an English-language model pre-trained with the supervision of the smallest version of Generative Pre-trained Transformer 2 (GPT-2). Like GPT-2, DistilGPT2 can be used to generate text.

## Setting up the DistilGPT2 model.
This snippet below sets up a text-generation pipeline using Hugging Face's transformers library.

Imports: Brings in Auto classes that automatically detect the correct architecture for your model.

Tokenizer: Loads the distilgpt2 tokenizer to convert raw text into numeric "tokens" the model can process.

Model: Loads the actual distilgpt2 model. Setting output_hidden_states=True is the "power move" here—it forces the model to expose its internal mathematical representations from every layer, rather than just the final prediction.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2", output_hidden_states=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The snippet below performs a simple Text Generation task in three steps:

Encoding: tokenizer.encode converts your string into numeric tensors that the model can process.

Generation: `model.generate` predicts the next tokens. Setting max_length=5 limits the total sequence (prompt + new words) to 5 tokens. `do_sample=False` uses Greedy Search, meaning the model always picks the single most likely next word.

Decoding: tokenizer.decode converts those numeric predictions back into human-readable text.

tokenizer.decode() turns those IDs back into the words you see on the screen.

Note: Since "The Shawshank" is already 4 tokens,
- `The`
- `Shaw`
- `sh`
- `ank`

This will only generate about 2 new tokens, as `max_length = 5`. Try other values too.

In [ ]:
text = "The Shawshank"
# text = "Stirling University"

# Tokenize the input string
input = tokenizer.encode(text, return_tensors="pt")

# Run the model
output = model.generate(input, max_length=6, do_sample=False)

# Print the output
print('\n',tokenizer.decode(output[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



 The Shawshank Redemption)


You can print the original output too, before decoding. The output variable itself is a PyTorch Tensor and it contains the token IDs of the output.

In [ ]:
# Print the token IDs (of the output)
output

tensor([[  464, 18193,  1477,   962, 34433,     8]])

## From words to vectors and back

You can also print the token IDs of the input. Convert the input text into tokens using `tokenizer`. The tokenizer converts the input into tokens.

In the below code, you have,  
Subword Tokenization: Since "Shawshank" isn't a single common English word like "apple," the tokenizer breaks it into smaller, manageable pieces (subwords) that it recognizes.

- "The" --> 464
- " Shaw" --> 18193
- "sh" --> 1477
- "ank" --> 962
    
Why `['input_ids']`?

When you call tokenizer(text), it actually returns a dictionary containing more than just the IDs. It also includes an attention mask (a series of 1s telling the model which tokens to focus on). By adding ['input_ids'] at the end, you are telling Python to "just give me the numeric codes and ignore the rest."

In [ ]:
# Print the input token ids
text = "The Shawshank"
input = tokenizer(text, return_tensors="pt")['input_ids']
input

tensor([[  464, 18193,  1477,   962]])

Now, you have the IDs of the inputs and the outputs.
But human read words and not IDs (numbers).
Can we convert these IDs back to words.
First, we will convert these IDs to tokens and then it is easy to merge those tokens.

In the below code, the `convert_ids_to_tokens` will convert the input IDs into token. So, you get a list of strings
`['The', 'ĠShaw', 'sh', 'ank']`

Note the Strange Ġ: This symobl represents a space. In the GPT-2 tokenizer, spaces are treated as part of the following word rather than separate tokens.

In [ ]:
tokenizer.convert_ids_to_tokens(input[0])

['The', 'ĠShaw', 'sh', 'ank']

## Breathe meaning into numbers (Embedding)

The code below accesses the Word Token Embedding layer of the GPT-2 model.

In the architecture of distilgpt2 (and the original GPT-2), `wte` stands for **Word Token Embeddings**. This is the very first layer of the transformer, acting as a massive "lookup table" or dictionary.
What it contains:

The Matrix: It is a weights matrix of size (vocab_size,n_embd). For DistilGPT2, that is usually (50257,768).

The Mapping: Every unique token ID (like 464 for "The") corresponds to a specific row in this matrix.

The Vector: Each row is a 768-dimensional vector—a list of numbers that represents the "meaning" of that token in a high-dimensional mathematical space.

Why it’s used:

Computers can't "do math" on the number 464 because the value itself is arbitrary. By passing the ID through wte, the model converts that static ID into a dense vector. These vectors are "learned" during training so that words with similar meanings or grammatical roles end up with similar numeric values in the vector space.
Common Use Case:

If you run model.transformer.wte(input), you are getting the raw embeddings for your tokens before any "thinking" (attention or feed-forward layers) has happened.

In [ ]:
# This is the embedding matrix of the model
model.transformer.wte # Dimensions are: (Number of tokens in vocabulary, dimension of model)

Embedding(50257, 768)

Now, you know that the word `The` corresponds to ID `464.
When you run this code, you are asking the model to look up the specific vector representation for the word "The."

Specifically, you are retrieving the 768-dimensional vector (the "embedding") that corresponds to the token ID 464 in the model's internal vocabulary.

In [ ]:
import torch
# Get the embedding vector of token # 464 ('The')
model.transformer.wte(torch.tensor(464))

tensor([-6.2649e-02, -4.4906e-02,  5.5888e-02, -5.4657e-02, -1.1713e-01,
        -7.2870e-02, -2.2326e-01, -3.2198e-03,  6.8535e-03,  2.3608e-02,
        -5.8700e-02,  4.4439e-02,  7.4774e-02, -1.3818e-02,  1.1873e-01,
        -5.1842e-02,  5.4415e-02,  5.5382e-02, -3.3126e-02,  1.1923e-01,
        -7.3283e-02,  2.6658e-02, -8.4261e-02,  5.7980e-02, -4.2860e-03,
        -4.0704e-02,  6.5652e-02, -6.6221e-02, -1.0232e-01,  3.1356e-02,
        -1.8873e-02,  2.7774e-02, -2.0423e-02,  1.1994e-01, -8.7572e-02,
        -1.0579e-01, -3.1816e-01,  9.5807e-02,  1.1588e-01, -4.2873e-02,
         1.3187e-01, -1.3457e-01, -1.0421e-01, -1.2150e-01,  9.2551e-02,
        -2.7394e-02,  3.1406e-02,  6.4891e-03,  1.2296e-01, -2.0581e-01,
        -6.3499e-02,  5.4726e-02,  6.0272e-02,  1.1968e-01,  6.4859e-02,
        -3.4885e-01, -5.6065e-02, -2.4721e-03,  3.7549e-03, -1.2817e-02,
        -8.4894e-02, -1.6269e-02,  8.2583e-02, -4.0134e-02, -2.1432e-01,
        -2.6889e-03,  1.5291e-02,  5.2090e-02,  2.9

Let's try completing one fun statement. You can run on other staements too. Try using different choices of max_length

In [ ]:
text = "The chicken didn't cross the road because it was"

# Tokenize the input string
input = tokenizer.encode(text, return_tensors="pt")

# Run the model
output = model.generate(input, max_length=20, do_sample=True)

# Print the output
print('\n',tokenizer.decode(output[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



 The chicken didn't cross the road because it was a good chicken for you.






## About this notebook
This notebook is copied from Jay Alammar github page.
The original notebook is available here https://github.com/jalammar/jalammar.github.io/blob/master/notebooks/Simple_Transformer_Language_Model.ipynb

Notes added by Hazrat.
